# cath_interface_secondary_structure_analysis

## Important files generated in this Analysis

cath_domain_boundary_label_asym_seq_ids.tsv
missing_cif_files.txt
domain_dssp_ss_summary.csv
get_domain_dssp_secondary_structure.py
cath_superfamily_ss_summary.csv
secondary_structure_fraction_boxplots_fig.png
secondary_structure_fraction_boxplots_fig.svg
secondary_structure_fraction_data.tsv
mixed_anova_results.tsv
pairwise_tests.tsv
notebook_for_codes.ipynb

## Map CATH domain boundaries to `label_asym_id` chains

This workflow converts the author chain identifiers (`auth_asym_id`) provided in the CATH domain boundary file to the corresponding `label_asym_id` chain identifiers used in DSSP mmCIF files. The resulting mapping is saved as a tab-separated file for downstream secondary structure analysis. 

In [ ]:
from pathlib import Path
from collections import defaultdict
import gemmi
from tqdm import tqdm


# ---------------- Configuration ---------------- #

# Input CATH domain boundaries (author chain IDs)
CATH_FILE = Path("/fsimb/groups/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/cath-domain-boundaries-seqreschopping.txt")

# Directory containing DSSP mmCIF files
DSSP_DIR = Path("/fsimb/groups/imb-luckgr/imb-luckgr2/projects/interface_clustering/dssp_download_20251030/cif")

# Output files
OUTPUT = Path("cath_domain_boundary_label_asym_seq_ids.tsv")
MISSING = Path("missing_cif_files.txt")


# ---------------- Read CATH domain boundaries ---------------- #

domains = defaultdict(list)

with open(CATH_FILE) as f:

    for line in f:

        if not line.strip():
            continue

        domain_id, boundary = line.split()

        pdb = domain_id[:4]
        auth_chain = domain_id[4]

        domains[pdb].append(
            (
                domain_id,
                auth_chain,
                boundary,
            )
        )


# ---------------- Map author chains to label chains ---------------- #

rows = []
missing = []


for pdb, domain_list in tqdm(domains.items(), total=len(domains)):

    cif = DSSP_DIR / pdb[1:3] / f"{pdb}.cif.gz"

    if not cif.exists():
        missing.append(str(cif))
        continue


    try:

        block = gemmi.cif.read_file(str(cif)).sole_block()

        table = block.find(
            "_struct_conf.",
            [
                "beg_label_asym_id",
                "beg_auth_asym_id",
            ],
        )

    except Exception:
        continue


    # Build a mapping from auth_asym_id to label_asym_id.
    # Only the first mapping is retained since chain identifiers are unique.
    auth_to_label = {}

    for row in table:

        label_chain = row[0]
        auth_chain = row[1]

        if label_chain == "?" or auth_chain == "?":
            continue

        # keep first mapping only
        if auth_chain not in auth_to_label:
            auth_to_label[auth_chain] = label_chain


    # Replace CATH author chain IDs with label_asym_id.
    for domain_id, auth_chain, boundary in domain_list:

        label_chain = auth_to_label.get(auth_chain)

        if label_chain is None:
            continue

        rows.append(
            (
                domain_id,
                label_chain,
                boundary,
            )
        )


# ---------------- Save results ---------------- #

with open(OUTPUT, "w") as f:

    #f.write("Domain\tLabel_asym_id\tBoundary\n")

    for domain, label_chain, boundary in rows:
        f.write(
            f"{domain}\t{label_chain}\t{boundary}\n"
        )


if missing:

    with open(MISSING, "w") as f:
        for item in sorted(set(missing)):
            #f.write(item + "\n")


print(f"Written {len(rows):,} domains to {OUTPUT}")

if missing:
    print(f"Missing CIF files: {len(set(missing))}")

## Calculate secondary structure composition of CATH domains

This workflow assigns DSSP secondary structure annotations to CATH domains by mapping CATH domain boundaries onto DSSP mmCIF residue annotations. For each CATH domain, it calculates the residue count and fractional composition of each secondary structure category, including unassigned residues.

In [ ]:
"""
Summarize DSSP secondary structure composition for CATH domains.

This script maps CATH domain boundaries onto DSSP secondary structure
assignments and calculates the number and percentage of residues in each
secondary structure class for every CATH domain.

Input
-----
1. cath_domain_boundary_label_asym_seq_ids.tsv
   Tab-separated file containing at least the following columns:
       Domain          CATH domain identifier (e.g. 1abcA00)
       Label_asym_id   Chain identifier matching DSSP label_asym_id
       Boundary        Domain boundaries in label_seq_id numbering
                       (e.g. 15-78 or 15-78,120-156)

2. DSSP mmCIF files
   One gzipped mmCIF file per PDB entry located under DSSP_DIR using
   the directory structure:
       <DSSP_DIR>/<pdb[1:3]>/<pdb>.cif.gz

Output
------
domain_ss_summary_3.csv
    One row per CATH domain containing:
        - residue counts for each DSSP secondary structure code
        - percentages of each secondary structure code
        - number of unassigned (NA) residues

missing_cif_files_3.txt
    List of DSSP files that could not be found.

Dependencies
------------
Python >=3.10
pandas
gemmi
tqdm
"""

from pathlib import Path
from collections import defaultdict, Counter
import pandas as pd
import gemmi
from tqdm import tqdm



#----------- Configuration ---------------#

CATH_FILE = Path("cath_domain_boundary_label_asym_seq_ids.tsv")
DSSP_DIR = Path("/fsimb/groups/imb-luckgr/imb-luckgr2/projects/interface_clustering/dssp_download_20251030/cif")
OUTPUT = Path("domain_dssp_ss_summary.csv")

MISSING_OUTPUT = Path("missing_cif_files_3.txt")
missing_cifs = []

# DSSP secondary structure codes
SS_CODES = ["H", "B", "E", "G", "I", "P", "T", "S"]
ALL_CODES = SS_CODES + ["NA"]


def load_domains(cath_file):

    """
    Read the CATH domain table and group domains by PDB entry.

    Returns
    -------
    dict
        {pdb_id: [(domain_id, chain, residue_set, domain_length), ...]}
    """

    domains = defaultdict(list)

    df = pd.read_csv(cath_file, sep="\t")

    # extract domain id, boundary, pdb id, chain
    for _, row in df.iterrows():
        domain_id = row["Domain"]
        chain = row["Label_asym_id"]
        boundary = row["Boundary"]
        pdb = domain_id[:4]
        residue_set = set()

        # Map domain boundary to start-end integers (compatible with discontinuous domains)
        for segment in boundary.split(","):
            start, end = map(int, segment.split("-"))
            residue_set.update(range(start, end + 1))

        domains[pdb].append((domain_id, chain, residue_set, len(residue_set),))

    return domains



def read_dssp(cif_file, required_chains):

    """
    Extract DSSP secondary structure assignments for selected chains.

    Only residues from the requested chains are parsed to reduce memory use.
    Unknown or missing DSSP codes are assigned as 'NA'.
    """

    # Read the DSSP summary table from the mmCIF file.
    block = gemmi.cif.read_file(str(cif_file)).sole_block()
    table = block.find(
        "_dssp_struct_summary.",
        [
            "label_asym_id",
            "label_seq_id",
            "secondary_structure",
        ],
    )

    residues = defaultdict(list)

    for row in table:
        chain = row[0]

        if chain not in required_chains:
            continue

        try:
            resnum = int(row[1])

        except ValueError:
            continue

        ss = row[2]
        if ss not in SS_CODES:
            ss = "NA"

        residues[chain].append((resnum, ss))

    return residues



def process_pdb(pdb, domains):

    """
    Calculate residue counts and percentages of DSSP secondary structure
    classes for every domain in a single PDB entry.
    """

    cif = (DSSP_DIR/ pdb[1:3] / f"{pdb}.cif.gz")

    if not cif.exists():
        missing_cifs.append(str(cif))
        return []

    required_chains = {chain for _, chain, _, _ in domains}
    residues = read_dssp(cif, required_chains)

    rows = []
    for domain_id, chain, residue_set, total in domains:

        counter = Counter()

        # Match DSSP label_seq_id residues to the CATH domain boundaries
        for resnum, ss in residues.get(chain, ()):
            if resnum in residue_set:
                counter[ss] += 1

        # Residues within the CATH domain that are absent from DSSP are
        # counted as unassigned (NA).
        counter["NA"] += (total - sum(counter.values()))

        row = {"Domain": domain_id, "Total": total, }

        for ss in ALL_CODES:
            row[ss] = counter[ss]

        for ss in ALL_CODES:
            row[f"{ss}%"] = (round(counter[ss] * 100 / total, 2) if total > 0 else 0.0)

        rows.append(row)

    return rows



def main():

    domains = load_domains(CATH_FILE)

    results = []

    for pdb, domain_list in tqdm(domains.items(), total=len(domains)):
        results.extend(process_pdb(pdb, domain_list))

    df = pd.DataFrame(results)
    df.to_csv(OUTPUT, index=False)
    print(df.head())

    if missing_cifs:
        with open(MISSING_OUTPUT, "w") as f:
            for item in sorted(set(missing_cifs)):
                f.write(item + "\n")

        print(f"Missing CIFs: {len(set(missing_cifs))}")

    print(f"\nSaved {len(df):,} domains to {OUTPUT}")


if __name__ == "__main__":
    main()

## Aggregate secondary structure composition by CATH homologous superfamily

This workflow combines per-domain secondary structure summaries with CATH classifications, groups domains by homologous superfamily, and calculates the median fractional abundance of each secondary structure category for every superfamily.

In [ ]:
from pathlib import Path
import pandas as pd


#----------C onfiguration --------------#

DOMAIN_SS = Path("domain_dssp_ss_summary.csv")
CATH_LIST = Path("/fsimb/groups/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/mapping/cath-domain-list.txt")
OUTPUT = Path("cath_superfamily_ss_summary.csv")


#---------- Read per-domain secondary structure summary --------------#

ss = pd.read_csv(
    DOMAIN_SS,
    usecols=[
        "Domain",
        "H%", "B%", "E%", "G%", "I%", "P%", "T%", "S%", "NA%"
    ],
)

# Combine DSSP secondary structure classes into broader categories
ss["helix_frac"] = ss["H%"] + ss["G%"] + ss["I%"] + ss["P%"]
ss["strand_frac"] = ss["E%"] + ss["B%"]
ss["loop_turn_frac"] = ss["T%"] + ss["S%"]
ss["NA_frac"] = ss["NA%"]

ss = ss[
    [
        "Domain",
        "helix_frac",
        "strand_frac",
        "loop_turn_frac",
        "NA_frac",
    ]
]

#---------- Read CATH classifications --------------#

cath = pd.read_csv(
    CATH_LIST,
    comment="#",
    sep=r"\s+",
    header=None,
    usecols=[0, 1, 2, 3, 4],
    names=[
        "Domain",
        "Class",
        "Architecture",
        "Topology",
        "HomologousSuperfamily",
    ],
)

# Construct the standard CATH superfamily identifier (C.A.T.H)
cath["Superfamily"] = (
    cath["Class"].astype(str)
    + "."
    + cath["Architecture"].astype(str)
    + "."
    + cath["Topology"].astype(str)
    + "."
    + cath["HomologousSuperfamily"].astype(str)
)

cath = cath[["Domain", "Superfamily"]]


#---------- Merge secondary structure and CATH data --------------#

merged = ss.merge(cath, on="Domain", how="inner")
print(f"Matched domains: {len(merged):,}")


#---------- Summarize by homologous superfamily --------------#

summary = (
    merged
    .groupby("Superfamily", sort=True)
    .agg(
        Domains=("Domain", lambda x: " ".join(sorted(x))),
        helix_frac=("helix_frac", "median"),
        strand_frac=("strand_frac", "median"),
        loop_turn_frac=("loop_turn_frac", "median"),
        NA_frac=("NA_frac", "median"),
    )
    .reset_index()
)

# Round values
summary[
    [
        "helix_frac",
        "strand_frac",
        "loop_turn_frac",
        "NA_frac",
    ]
] = summary[
    [
        "helix_frac",
        "strand_frac",
        "loop_turn_frac",
        "NA_frac",
    ]
].round(2)

#---------- Save results --------------#
summary.to_csv(OUTPUT, index=False)
print(summary.head())
print(f"\nSaved {len(summary):,} superfamilies to {OUTPUT}")

## Visualize secondary structure distributions across interface types and CATH superfamilies

This workflow combines secondary structure fractions from ordered interfaces, disordered interfaces, and CATH homologous superfamilies into a common format and generates comparative boxplots for each secondary structure category.

In [28]:
from pathlib import Path
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt


# ---------------- Configuration ---------------- #

CLUSTER_FILE = Path("/Volumes/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_cluster_summary.tsv")
SUPERFAMILY_FILE = Path("/Volumes/imb-luckgr/projects/interface_clustering/datasets/cluster_analysis/cath_superfamily_ss_summary.csv")

OUT_PNG = "/Volumes/imb-luckgr/projects/interface_clustering/visualizations/plots/secondary_structure_fraction_boxplots_fig.png"
OUT_PDF = "/Volumes/imb-luckgr/projects/interface_clustering/visualizations/plots/secondary_structure_fraction_boxplots_fig.pdf"


# ---------------- Load datasets ---------------- #

clusters = pd.read_csv(CLUSTER_FILE, sep="\t")

ordered = clusters[clusters["top_discat"] == "Order-order"].copy()
disordered = clusters[clusters["top_discat"] == "Disorder-disorder"].copy()

sf = pd.read_csv(SUPERFAMILY_FILE)

for c in ["helix_frac", "strand_frac", "loop_turn_frac", "NA_frac"]:
    sf[c] = sf[c] / 100.0


# ---------------- Assemble plotting data  ---------------- #

mapping = {
    "Helix": ("median_helixfrac", "helix_frac"),
    "Strand": ("median_betafrac", "strand_frac"),
    "Turn / Loop": ("median_bendturnfrac", "loop_turn_frac"),
    "Unassigned": ("median_unassignfrac", "NA_frac"),
}

records = []

for ss, (c_col, s_col) in mapping.items():

    for v in ordered[c_col].dropna():
        records.append({
            "SS": ss,
            "Source": "Ordered interfaces",
            "Fraction": v
        })

    for v in disordered[c_col].dropna():
        records.append({
            "SS": ss,
            "Source": "Disordered interfaces",
            "Fraction": v
        })

    for v in sf[s_col].dropna():
        records.append({
            "SS": ss,
            "Source": "CATH superfamilies",
            "Fraction": v
        })

plot_df = pd.DataFrame(records)

# ---------------- Plot style ---------------- #

sns.set_style("ticks")

plt.rcParams.update({
    "font.family": "Arial",

})

# muted, Nature-style palette
palette = {
    "Ordered interfaces": "#4C78A8",     # muted blue
    "Disordered interfaces": "#E45756",  # muted red
    "CATH superfamilies": "#72B7B2",     # muted teal
}


# ---------------- Generate boxplots ---------------- #

fig, ax = plt.subplots(figsize=(3, 3))  # slim subfigure size

sns.boxplot(
    data=plot_df,
    x="SS",
    y="Fraction",
    hue="Source",
    palette=palette,
    width=0.5,               
    linewidth=0.6,           
    showfliers=False,
    whis=(5, 95),
    ax=ax,
    boxprops=dict(edgecolor="0.25"),
    medianprops=dict(color="black", linewidth=1.6),
    whiskerprops=dict(linewidth=0.9),
    capprops=dict(linewidth=0.9),
)

ax.set_xlabel("")
ax.set_ylabel(
    "Residue median fraction in\nsecondary structure class",
    fontsize=12
)

ax.set_ylim(0, 1)

ax.set_xticks([0,1,2,3])
ax.set_xticklabels([
    "Helix",
    "Strand",
    "Turn",
    "Unassigned"
])
ax.tick_params(axis='both', labelsize=8)

ax.legend(
    frameon=False,
    loc="upper right",
    fontsize=8
)

ax.grid(False)
ax.spines.top.set_visible(False)
ax.spines.right.set_visible(False)

plt.tight_layout()


# ---------------- Save figure ---------------- #

plt.savefig(OUT_PNG, dpi=300, bbox_inches="tight")
plt.savefig(OUT_PDF, bbox_inches="tight")

plt.close()

print("Saved:")
print(OUT_PNG)
print(OUT_PDF)

Saved:
/Volumes/imb-luckgr/projects/interface_clustering/visualizations/plots/secondary_structure_fraction_boxplots_fig.png
/Volumes/imb-luckgr/projects/interface_clustering/visualizations/plots/secondary_structure_fraction_boxplots_fig.pdf


In [24]:
plot_df["SS"].unique()

array(['Helix', 'Strand', 'Turn / Loop', 'Unassigned'], dtype=object)

## Prepare secondary structure fractions for statistical analysis

This workflow combines secondary structure fractions from ordered interfaces, disordered interfaces, and CATH homologous superfamilies into a long-format table suitable for statistical analysis and visualization. Each observation is assigned a unique identifier, group label, secondary structure category, and fractional abundance.

In [ ]:
from pathlib import Path
import pandas as pd

# ---------------- Configuration ---------------- #

CLUSTER_FILE = Path("/fsimb/groups/imb-luckgr/projects/interface_clustering/results/cluster_analysis/pdb_cluster_summary.tsv")
SUPERFAMILY_FILE = Path("cath_superfamily_ss_summary.csv")
OUTPUT = Path("secondary_structure_fraction_data.tsv")

# ---------------- Load input datasets ---------------- #

clusters = pd.read_csv(CLUSTER_FILE, sep="\t")

# Select clusters containing only ordered-order and disorder-disorder interfaces
ordered = clusters[clusters["top_discat"] == "Order-order"].reset_index(drop=True)
disordered = clusters[clusters["top_discat"] == "Disorder-disorder"].reset_index(drop=True)

sf = pd.read_csv(SUPERFAMILY_FILE)

# Convert CATH superfamily secondary structure values from percentages to fractions
for c in ["helix_frac", "strand_frac", "loop_turn_frac", "NA_frac"]:
    sf[c] /= 100.0

# Map secondary structure categories to corresponding columns in each dataset
mapping = {
    "Helix": ("median_helixfrac", "helix_frac"),
    "Strand": ("median_betafrac", "strand_frac"),
    "Turn / Loop": ("median_bendturnfrac", "loop_turn_frac"),
    "Unassigned": ("median_unassignfrac", "NA_frac"),
}

records = []

# Ordered interface clusters
for i, row in ordered.iterrows():

    ID = f"O_{i+1}"

    for ss, (cluster_col, _) in mapping.items():

        records.append({
            "ID": ID,
            "Group": "Ordered interfaces",
            "Secondary_structure": ss,
            "Fraction": row[cluster_col]
        })

# Disordered interface clusters
for i, row in disordered.iterrows():

    ID = f"D_{i+1}"

    for ss, (cluster_col, _) in mapping.items():

        records.append({
            "ID": ID,
            "Group": "Disordered interfaces",
            "Secondary_structure": ss,
            "Fraction": row[cluster_col]
        })

# CATH homologous superfamilies
for i, row in sf.iterrows():

    ID = f"SF_{i+1}"

    for ss, (_, sf_col) in mapping.items():

        records.append({
            "ID": ID,
            "Group": "CATH superfamilies",
            "Secondary_structure": ss,
            "Fraction": row[sf_col]
        })

# ---------------- Save formatted dataset ---------------- #

plot_df = pd.DataFrame(records)

plot_df.to_csv(OUTPUT, sep="\t", index=False)

print(plot_df.head())
print(f"\nSaved {len(plot_df):,} rows to {OUTPUT}")

## Statistical comparison of secondary structure distributions

This workflow performs a mixed ANOVA to compare secondary structure fractions across groups, followed by post hoc pairwise t-tests with Holm correction for multiple testing. The statistical results are saved as tab-separated files.

In [ ]:
import pandas as pd
import pingouin as pg

# ---------------- Load input data ---------------- #

df = pd.read_csv("secondary_structure_fraction_data.tsv", sep="\t")

# ---------------- Mixed ANOVA ---------------- #

anova = pg.mixed_anova(
    data=df,
    dv="Fraction",
    between="Group",
    within="Secondary_structure",
    subject="ID",
)

print("\nMixed ANOVA\n")
print(anova)

anova.to_csv(
    "mixed_anova_results.tsv",
    sep="\t",
    index=False
)

# ---------------- Post hoc pairwise comparisons ---------------- #

pairwise = pg.pairwise_tests(
    data=df,
    dv="Fraction",
    between="Group",
    within="Secondary_structure",
    subject="ID",
    padjust="holm"
)

pairwise.to_csv(
    "pairwise_tests.tsv",
    sep="\t",
    index=False
)

print("\nSaved:")
print("mixed_anova_results.tsv")
print("pairwise_tests.tsv")